In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate


c:\Users\kr070\Desktop\RAG_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kr070\AppData\Local\Temp\ipykernel_11984\3030831507.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


##### step 1a. Extract the transcript of the Youtube video

In [2]:
video_id = "E8zpgNPx8jE"

try: 
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No caption available for this video")

So, in this video, I'm going to show you something I've never done on this channel before, and that is an end to end agent build. Literally from start to finish, from ideation all the way to deployment, and this actually running on a server and working, you can follow along and learn from this. And what we're going to build is we're going to create an AI news aggregator. So, this is going to be super exciting. So, what this will do once you have this deployed, which I will show you in this video, every day you will get an email with the latest AI news specifically tailored to you. We can specify YouTube channels that we want to follow, blog posts, RSS feeds. We'll bring all of that together in a scraping pipeline that will run every day, and we'll use AI and language models to tailor it specifically for us based on our interests. And not only that, I'm going to walk you through everything. The deployment, how I use AI assisted coding tools in order to plan, architect, and design new fe

##### Step 1b Create chunks of the extracted Transcript

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, 
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript])

print(len(chunks))
print("\nFirst Chunk : ", chunks[0].page_content)

193

First Chunk :  So, in this video, I'm going to show you something I've never done on this channel before, and that is an end to end agent build. Literally from start to finish, from ideation all the way to deployment, and this actually running on a server and working, you can follow along and learn from this. And what we're going to build is we're going to create an AI news aggregator. So, this is going to be super exciting. So, what this will do once you have this deployed, which I will show you in this video, every day you will get an email with the latest AI news specifically tailored to you. We can specify YouTube channels that we want to follow, blog posts, RSS feeds. We'll bring all of that together in a scraping pipeline that will run every day, and we'll use AI and language models to tailor it specifically for us based on our interests. And not only that, I'm going to walk you through everything. The deployment, how I use AI assisted coding tools in order to plan, architec

Indexing (Embedding Generation and storing transcripts in Vector Stores)

In [4]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2421.78it/s]


Forming Retriever 

In [5]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [7]:
retriever.invoke('What is the project all about ?')

[Document(id='beac4c42-18a8-4b58-9970-f9f03250fa05', metadata={}, page_content="specifically for us based on our interests. And not only that, I'm going to walk you through everything. The deployment, how I use AI assisted coding tools in order to plan, architect, and design new features, how to bring everything together in a Docker Compose file, how to deploy it via render, how to set up and manage the database, literally everything. so that by the end, like I've said, you'll not only have a serious portfolio project, but also you'll get a chance to understand and get a behind-the-scenes view of how I structure these projects. I'm going to show you everything, how I debug, how I plan, how I run into all of the challenges. So, that's going to be the most valuable part of this video beyond just creating the news aggregator. So, if you're serious about learning AI and you really want to get a deep understanding of what AI engineering and building projects like this look in the real world

Augmentation 

In [8]:
model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.7
)

In [9]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. 
    Answer ONLY from the provided transcript context.
    Please provide the answer in a bit detail
    If the context is insufficient just say you don't know. 
    {context}
    question : {question}
    """,
    input_variables=['question', 'context']
)

In [42]:
question = "The host deploys the project on Render, can i also do it if i have a free plan of render and use it the same way ?"
retrieved_docs = retriever.invoke(question)

In [43]:
# Retrieves the embeddings which are most similar to question embedding
retrieved_docs

[Document(id='93b1fa73-b7f3-4c06-ad61-1899dea32286', metadata={}, page_content="we're good on this side. And then I'm going to say, okay, every awesome everything now works. How what would now be the simplest way to put this on a server something like I'm considering render but I'm not sure if it's the best one to package this and deploy it where we have the database deployed as well and within the docker container and we schedule the main.py file to run every 24 hours. First let's create a plan to do this. Let's actually get back to planning mode because I there's so many ways to do this, right? So, I'm I'm just trying to see like what what a quick quick and easy way to do this is. Okay, so it came up with a plan. This is going to get tricky, but we'll we'll see what we can do here. I've never worked with render, so I know that render just an easy way where you you employ this deploy stuff, but of course, every cloud provider still has its own like nuances there. But we have a solid w

concatinating all the retrieved docs to create context for the llm

In [44]:
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
context

"we're good on this side. And then I'm going to say, okay, every awesome everything now works. How what would now be the simplest way to put this on a server something like I'm considering render but I'm not sure if it's the best one to package this and deploy it where we have the database deployed as well and within the docker container and we schedule the main.py file to run every 24 hours. First let's create a plan to do this. Let's actually get back to planning mode because I there's so many ways to do this, right? So, I'm I'm just trying to see like what what a quick quick and easy way to do this is. Okay, so it came up with a plan. This is going to get tricky, but we'll we'll see what we can do here. I've never worked with render, so I know that render just an easy way where you you employ this deploy stuff, but of course, every cloud provider still has its own like nuances there. But we have a solid we have a solid plan. So, I'm first going to ask a couple of things and then I'm

In [45]:
final_prompt = prompt.invoke({"question": question, "context": context})
print(final_prompt)

text="\n    You are a helpful assistant. \n    Answer ONLY from the provided transcript context.\n    Please provide the answer in a bit detail\n    If the context is insufficient just say you don't know. \n    we're good on this side. And then I'm going to say, okay, every awesome everything now works. How what would now be the simplest way to put this on a server something like I'm considering render but I'm not sure if it's the best one to package this and deploy it where we have the database deployed as well and within the docker container and we schedule the main.py file to run every 24 hours. First let's create a plan to do this. Let's actually get back to planning mode because I there's so many ways to do this, right? So, I'm I'm just trying to see like what what a quick quick and easy way to do this is. Okay, so it came up with a plan. This is going to get tricky, but we'll we'll see what we can do here. I've never worked with render, so I know that render just an easy way wher

In [46]:
answer = model.invoke(final_prompt)
print(answer.content)

I don’t know.


### Building the chain 

In [47]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [48]:
#generating a function for the concatination of the retrieved docs to pass in the parallele chain using RunnableLambda
def Format_docs(retrieved_docs):
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context

In [49]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(Format_docs),
    'question': RunnablePassthrough()
})

In [50]:
parallel_chain.invoke("The host deploys the project on Render, can i also do it if i have a free plan of render and use it the same way ?")

{'context': "we're good on this side. And then I'm going to say, okay, every awesome everything now works. How what would now be the simplest way to put this on a server something like I'm considering render but I'm not sure if it's the best one to package this and deploy it where we have the database deployed as well and within the docker container and we schedule the main.py file to run every 24 hours. First let's create a plan to do this. Let's actually get back to planning mode because I there's so many ways to do this, right? So, I'm I'm just trying to see like what what a quick quick and easy way to do this is. Okay, so it came up with a plan. This is going to get tricky, but we'll we'll see what we can do here. I've never worked with render, so I know that render just an easy way where you you employ this deploy stuff, but of course, every cloud provider still has its own like nuances there. But we have a solid we have a solid plan. So, I'm first going to ask a couple of things 

In [51]:
parser = StrOutputParser()

In [52]:
chain = parallel_chain | prompt | model | parser

In [53]:
chain.invoke("The host deploys the project on Render, can i also do it if i have a free plan of render and use it the same way and use it for free?")

'I don’t have enough information in the transcript to say whether you can deploy the project on Render using a free plan and run it the same way without cost. The conversation discusses the steps for setting up the deployment, but it doesn’t mention the limitations or capabilities of Render’s free tier.'